# 🚀 AI Image & Video Upscaler — Google Colab (GPU Gratis)

Notebook ini menjalankan upscaling gambar/video pakai **Real-ESRGAN**, sama seperti versi lokal, tapi pakai GPU Colab (biasanya jauh lebih cepat & VRAM lebih besar daripada RTX 3050 laptop).

**Cara pakai:**
1. Pastikan GPU aktif: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → Save
2. Jalankan semua cell dari atas ke bawah **secara berurutan** (klik ▶ di tiap cell, atau `Shift+Enter`)
3. Di cell "Upload File", kamu akan diminta upload gambar/video dari komputer
4. Di cell "Atur Opsi", sesuaikan scale/mode sebelum lanjut ke cell "Jalankan"
5. Di akhir, hasil bisa langsung didownload

⏱️ Sesi Colab gratis biasanya maksimal ~12 jam dan bisa terputus kalau browser idle lama — untuk video panjang, pastikan tab tetap terbuka & aktif.

## 1. Cek GPU

In [ ]:
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU aktif: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {vram_gb:.1f} GB")
else:
    print("⚠️  GPU belum aktif!")
    print("    Buka menu: Runtime > Change runtime type > pilih 'T4 GPU' > Save")
    print("    Lalu jalankan ulang cell ini (Runtime > Restart session kalau perlu).")

## 2. Install Dependencies (±1-2 menit, sekali per sesi)

In [ ]:
!pip install -q realesrgan basicsr gfpgan facexlib
!apt-get -y -qq install ffmpeg > /dev/null
print("[✓] Instalasi selesai.")

## 3. Patch Kompatibilitas (wajib, sekali jalan)

Package `basicsr` masih mereferensikan modul lama `torchvision.transforms.functional_tensor`
yang sudah dihapus di versi torchvision terbaru (bug yang sama yang kita temui di laptop lokal).
Cell ini otomatis memperbaikinya di semua file yang terpengaruh.

In [ ]:
import os
import importlib.util

def patch_basicsr_torchvision_compat():
    # PENTING: cari lokasi basicsr TANPA mengimpornya langsung — import basicsr
    # akan otomatis menjalankan __init__.py yang justru memicu bug yang mau kita
    # perbaiki (ayam-telur). find_spec() cuma mencari lokasi file, tidak menjalankan
    # kodenya.
    spec = importlib.util.find_spec("basicsr")
    if spec is None or spec.origin is None:
        print("[x] Tidak bisa menemukan lokasi package basicsr.")
        return []

    base_dir = os.path.dirname(spec.origin)
    patched_files = []
    for root, _, filenames in os.walk(base_dir):
        for fname in filenames:
            if not fname.endswith(".py"):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            if "torchvision.transforms.functional_tensor" in content:
                content = content.replace(
                    "torchvision.transforms.functional_tensor",
                    "torchvision.transforms.functional",
                )
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(content)
                patched_files.append(fpath)
    return patched_files

patched = patch_basicsr_torchvision_compat()
print(f"[✓] {len(patched)} file di-patch untuk kompatibilitas torchvision terbaru.")

## 4. Mesin Upscaling (jalankan sekali, tidak perlu diubah)

In [ ]:
import time
import subprocess
import tempfile
import warnings
from pathlib import Path

import cv2
import torch
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

MODEL_INFO = {
    2: {"name": "RealESRGAN_x2plus",
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
        "netscale": 2},
    4: {"name": "RealESRGAN_x4plus",
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "netscale": 4},
}

# Model "cepat": arsitektur SRVGGNetCompact, 2-4x lebih cepat dari RRDB di atas,
# kualitas sedikit di bawah tapi masih bagus. Cocok untuk video (banyak frame).
FAST_MODEL_INFO = {
    "name": "realesr-general-x4v3 (Mode Cepat)",
    "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth",
    "netscale": 4,
}

SUPPORTED_IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SUPPORTED_VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv", ".webm"}


def build_upsampler(scale=4, tile=0, half=True, fast=False):
    from realesrgan import RealESRGANer

    if fast:
        from basicsr.archs.srvgg_arch import SRVGGNetCompact
        info = FAST_MODEL_INFO
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=32,
                                 upscale=info["netscale"], act_type="prelu")
    else:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        info = MODEL_INFO[scale]
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23,
                         num_grow_ch=32, scale=info["netscale"])

    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0

    # GPU Colab biasanya VRAM-nya jauh lebih besar dari laptop, jadi tile bisa dibikin
    # lebih besar (lebih sedikit overhead potong-gabung). TAPI tetap TIDAK pernah benar-benar
    # dimatikan (tile=0) — frame video/gambar sumber bisa jadi sangat besar (mis. video portrait
    # resolusi tinggi), yang tetap bisa membuat OOM walau VRAM-nya besar kalau diproses utuh
    # tanpa dipotong sama sekali.
    if tile == 0:
        if vram_gb >= 14:
            tile = 800
        elif vram_gb >= 10:
            tile = 600
        else:
            tile = {2: 512, 4: 256}.get(scale, 400)
        tile_note = f"{tile} (auto, VRAM {vram_gb:.1f}GB)"
    else:
        tile_note = f"{tile} (manual)"

    upsampler = RealESRGANer(
        scale=info["netscale"],
        model_path=info["url"],
        model=model,
        tile=tile,
        tile_pad=10,
        pre_pad=0,
        half=half and torch.cuda.is_available(),
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    print(f"[i] Model: {info['name']} | tile={tile_note} | half={half}")
    return upsampler


def sharpen_image(img, amount=1.0, radius=1.5):
    blurred = cv2.GaussianBlur(img, (0, 0), radius)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def build_face_enhancer():
    from gfpgan import GFPGANer
    model_path = "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"
    print("[i] Face enhancement (GFPGAN) aktif.")
    return GFPGANer(model_path=model_path, upscale=1, arch="clean", channel_multiplier=2, bg_upsampler=None)


def upscale_frame(upsampler, img, outscale, face_enhancer=None, sharpen=False, sharpen_amount=1.0,
                   retry_tiles=(256, 128, 64)):
    try:
        output, _ = upsampler.enhance(img, outscale=outscale)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        output = None
        for t in retry_tiles:
            print(f"[!] CUDA out of memory, mencoba ulang dengan tile={t}...")
            upsampler.tile_size = t
            try:
                output, _ = upsampler.enhance(img, outscale=outscale)
                break
            except RuntimeError:
                torch.cuda.empty_cache()
                continue
        if output is None:
            raise

    if face_enhancer is not None:
        torch.cuda.empty_cache()
        try:
            _, _, output = face_enhancer.enhance(output, has_aligned=False, only_center_face=False, paste_back=True)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print("[!] GFPGAN kehabisan VRAM, dilewati untuk frame ini.")
            else:
                raise

    if sharpen:
        output = sharpen_image(output, amount=sharpen_amount)
    return output


def upscale_one(upsampler, in_path, out_path, outscale, face_enhancer=None, sharpen=False, sharpen_amount=1.0):
    img = cv2.imread(str(in_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"[x] Gagal membaca gambar: {in_path}")
        return False
    output = upscale_frame(upsampler, img, outscale, face_enhancer, sharpen, sharpen_amount)
    out_path = Path(out_path)
    if out_path.suffix.lower() not in SUPPORTED_IMAGE_EXT:
        out_path = out_path.with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), output)
    print(f"[✓] Disimpan: {out_path}")
    return True


def upscale_video(upsampler, in_path, out_path, outscale, face_enhancer=None, sharpen=False, sharpen_amount=1.0):
    cap = cv2.VideoCapture(str(in_path))
    if not cap.isOpened():
        print(f"[x] Gagal membuka video: {in_path}")
        return False

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[i] Video: {total_frames} frame @ {fps:.2f}fps")

    ffmpeg_proc = None
    stderr_file = None
    frame_idx = 0
    t_first = None

    pbar = tqdm(total=total_frames if total_frames > 0 else None, desc="Upscaling video")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx == 0:
                t_first = time.time()

            out_frame = upscale_frame(upsampler, frame, outscale, face_enhancer, sharpen, sharpen_amount)

            if frame_idx == 0:
                elapsed = time.time() - t_first
                est_total_min = (elapsed * total_frames) / 60
                print(f"[i] Frame pertama selesai dalam {elapsed:.1f} detik.")
                print(f"[i] Perkiraan total waktu: ~{est_total_min:.1f} menit untuk {total_frames} frame.")

            if ffmpeg_proc is None:
                out_h, out_w = out_frame.shape[:2]
                print(f"[i] Resolusi output: {out_w}x{out_h}")
                cmd = [
                    "ffmpeg", "-y",
                    "-f", "rawvideo", "-pix_fmt", "bgr24",
                    "-s", f"{out_w}x{out_h}", "-r", str(fps),
                    "-i", "-",                       # input 1: frame mentah dari stdin
                    "-i", str(in_path),               # input 2: file asli, cuma diambil audionya
                    "-map", "0:v:0", "-map", "1:a:0?",
                    "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
                    "-c:a", "aac", "-shortest",
                    str(out_path),
                ]
                # stderr ditampung ke FILE (bukan pipe) supaya tidak deadlock kalau
                # ffmpeg menulis banyak log progress sementara kita masih kirim frame.
                stderr_file = tempfile.TemporaryFile(mode="w+b")
                ffmpeg_proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=stderr_file)

            ffmpeg_proc.stdin.write(out_frame.tobytes())
            frame_idx += 1
            pbar.update(1)
    finally:
        cap.release()
        pbar.close()

    if frame_idx == 0 or ffmpeg_proc is None:
        print("[x] Tidak ada frame yang berhasil diproses.")
        return False

    ffmpeg_proc.stdin.close()
    ffmpeg_proc.wait()
    ok = ffmpeg_proc.returncode == 0
    if not ok:
        stderr_file.seek(0)
        print("[x] ffmpeg gagal encode:")
        print(stderr_file.read().decode(errors="ignore")[-800:])
    stderr_file.close()

    if ok:
        print(f"[✓] Video selesai: {out_path}")
    return ok


print("[✓] Mesin upscaling siap dipakai.")

## 5. Upload File

Jalankan cell di bawah, lalu pilih gambar/video dari komputermu lewat jendela yang muncul.

In [ ]:
import os
from pathlib import Path
from google.colab import files

print("Pilih file gambar atau video dari komputermu...")
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
input_path = Path(input_filename)
print(f"\n[✓] Terupload: {input_path} ({os.path.getsize(input_path) / 1e6:.1f} MB)")

### Checkpoint: verifikasi file sebelum lanjut

In [ ]:
is_video = input_path.suffix.lower() in SUPPORTED_VIDEO_EXT

if is_video:
    cap = cv2.VideoCapture(str(input_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"📹 Video: {w}x{h}, {total_frames} frame, {fps:.1f} fps (~{total_frames / fps:.1f} detik)")
else:
    from PIL import Image
    im = Image.open(input_path)
    print(f"🖼️  Gambar: {im.size[0]}x{im.size[1]}")

## 6. Atur Opsi Upscale

Sesuaikan nilai di bawah sebelum lanjut ke cell "Jalankan Upscaling".

In [ ]:
SCALE = 2              # 2 atau 4
FAST_MODE = True        # True = jauh lebih cepat (model ringan), False = kualitas maksimal
CUSTOM_OUTSCALE = None  # isi angka mis. 6 kalau mau resolusi akhir beda dari SCALE, atau None
SHARPEN = False
SHARPEN_AMOUNT = 1.0
FACE_ENHANCE = False    # True kalau isinya foto/video wajah orang

print("Opsi tersimpan:")
print(f"  SCALE={SCALE}, FAST_MODE={FAST_MODE}, OUTSCALE={CUSTOM_OUTSCALE or SCALE}, "
      f"SHARPEN={SHARPEN} ({SHARPEN_AMOUNT}), FACE_ENHANCE={FACE_ENHANCE}")

## 7. Jalankan Upscaling

⏱️ Untuk video: perkiraan waktu total akan muncul setelah frame pertama selesai diproses.

In [ ]:
outscale = CUSTOM_OUTSCALE or SCALE

upsampler = build_upsampler(scale=SCALE, fast=FAST_MODE)
face_enhancer = build_face_enhancer() if FACE_ENHANCE else None

ext = ".mp4" if is_video else input_path.suffix
output_path = input_path.with_name(f"{input_path.stem}_hd{ext}")

if is_video:
    success = upscale_video(upsampler, input_path, output_path, outscale, face_enhancer, SHARPEN, SHARPEN_AMOUNT)
else:
    success = upscale_one(upsampler, input_path, output_path, outscale, face_enhancer, SHARPEN, SHARPEN_AMOUNT)

print("\n[✓] SELESAI!" if success else "\n[x] GAGAL, cek pesan error di atas.")

### Preview hasil (khusus gambar)

In [ ]:
if not is_video and success:
    from IPython.display import Image as IPImage, display
    print("Sebelum:")
    display(IPImage(filename=str(input_path), width=320))
    print("Sesudah (HD):")
    display(IPImage(filename=str(output_path), width=320))
else:
    print("Preview inline dilewati untuk video (filenya terlalu besar) — langsung download saja di cell berikutnya.")

## 8. Download Hasil

In [ ]:
if success:
    files.download(str(output_path))
else:
    print("Tidak ada file untuk didownload karena proses gagal.")

### (Opsional) Simpan ke Google Drive

Kalau file hasil besar dan `files.download()` di atas gagal/lambat, pakai cara ini sebagai
alternatif — hapus tanda `#` di setiap baris untuk mengaktifkan, lalu jalankan cell-nya.

In [ ]:
# from google.colab import drive
# import shutil
#
# drive.mount('/content/drive')
# tujuan = f"/content/drive/MyDrive/{output_path.name}"
# shutil.copy(output_path, tujuan)
# print("Tersimpan di Google Drive:", tujuan)

---
### Tips
- Mau proses file lain? Cukup jalankan ulang dari cell **"5. Upload File"** — tidak perlu install ulang.
- Mau ganti opsi (scale/mode)? Edit cell **"6. Atur Opsi"**, lalu jalankan ulang cell **"7. Jalankan Upscaling"**.
- Kalau sesi Colab terputus (idle lama), jalankan ulang dari cell **"2. Install Dependencies"**.